In [2]:
import pandas as pd

df = pd.read_csv("../data/2019-Nov.csv", nrows=500000)

print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
df.head()

(500000, 9)
event_time        object
event_type        object
product_id         int64
category_id        int64
category_code     object
brand             object
price            float64
user_id            int64
user_session      object
dtype: object
event_time            0
event_type            0
product_id            0
category_id           0
category_code    162372
brand             74791
price                 0
user_id               0
user_session          0
dtype: int64


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [3]:
# ---- CLEANING ----

# 1. Fix event_time to datetime
df['event_time'] = pd.to_datetime(df['event_time'], utc=True)

# 2. Extract useful time columns
df['date'] = df['event_time'].dt.date
df['hour'] = df['event_time'].dt.hour
df['day_of_week'] = df['event_time'].dt.day_name()

# 3. Fill missing category_code and brand
df['category_code'] = df['category_code'].fillna('unknown')
df['brand'] = df['brand'].fillna('unknown')

# 4. Split category_code into levels
df[['cat_level1', 'cat_level2', 'cat_level3']] = df['category_code'].str.split('.', expand=True, n=2).reindex(columns=[0,1,2])
df['cat_level1'] = df['cat_level1'].fillna('unknown')
df['cat_level2'] = df['cat_level2'].fillna('unknown')
df['cat_level3'] = df['cat_level3'].fillna('unknown')

# 5. Verify
print(df.shape)
print(df.isnull().sum())
df.head(3)

(500000, 15)
event_time       0
event_type       0
product_id       0
category_id      0
category_code    0
brand            0
price            0
user_id          0
user_session     0
date             0
hour             0
day_of_week      0
cat_level1       0
cat_level2       0
cat_level3       0
dtype: int64


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,date,hour,day_of_week,cat_level1,cat_level2,cat_level3
0,2019-11-01 00:00:00+00:00,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33,2019-11-01,0,Friday,electronics,smartphone,unknown
1,2019-11-01 00:00:00+00:00,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283,2019-11-01,0,Friday,appliances,sewing_machine,unknown
2,2019-11-01 00:00:01+00:00,view,17302664,2053013553853497655,unknown,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387,2019-11-01,0,Friday,unknown,unknown,unknown


In [4]:
# ---- EDA ----

# 1. Event type distribution
print("=== Event Type Counts ===")
print(df['event_type'].value_counts())
print()

# 2. Conversion funnel
total = len(df)
views = len(df[df['event_type'] == 'view'])
carts = len(df[df['event_type'] == 'cart'])
purchases = len(df[df['event_type'] == 'purchase'])

print("=== Conversion Funnel ===")
print(f"Views:     {views} ({views/total*100:.1f}%)")
print(f"Carts:     {carts} ({carts/total*100:.1f}%)")
print(f"Purchases: {purchases} ({purchases/total*100:.1f}%)")
print(f"Cart → Purchase Rate: {purchases/carts*100:.2f}%")
print()

# 3. Top 10 brands by views
print("=== Top 10 Brands (Views) ===")
print(df[df['event_type']=='view']['brand'].value_counts().head(10))
print()

# 4. Top 5 categories
print("=== Top 5 Categories ===")
print(df[df['event_type']=='view']['cat_level1'].value_counts().head(5))
print()

# 5. Hourly traffic
print("=== Hourly Traffic ===")
print(df.groupby('hour')['event_type'].count().sort_index())

=== Event Type Counts ===
event_type
view        482642
purchase      9595
cart          7763
Name: count, dtype: int64

=== Conversion Funnel ===
Views:     482642 (96.5%)
Carts:     7763 (1.6%)
Purchases: 9595 (1.9%)
Cart → Purchase Rate: 123.60%

=== Top 10 Brands (Views) ===
brand
unknown     73913
samsung     57015
apple       45838
xiaomi      34611
huawei      11111
lucente      7495
oppo         6273
cordiant     6107
lg           5995
bosch        5705
Name: count, dtype: int64

=== Top 5 Categories ===
cat_level1
electronics    175560
unknown        158949
appliances      52849
computers       27573
furniture       22743
Name: count, dtype: int64

=== Hourly Traffic ===
hour
0    10887
1    14000
2    32498
3    49348
4    61480
5    73450
6    76155
7    79426
8    79818
9    22938
Name: event_type, dtype: int64


In [5]:
# Save cleaned data for Power BI
df.to_csv("../data/cleaned_nov2019.csv", index=False)
print("Saved! Rows:", len(df))

Saved! Rows: 500000


In [6]:
from sqlalchemy import create_engine

# Create SQLite database
engine = create_engine('sqlite:///../data/analytics.db')

# Push cleaned dataframe to SQL table
df.to_sql('ecommerce_events', con=engine, if_exists='replace', index=False)

print("Data pushed to SQLite!")

# Verify
import pandas as pd
check = pd.read_sql("SELECT COUNT(*) as total_rows FROM ecommerce_events", engine)
print(check)

Data pushed to SQLite!
   total_rows
0      500000


In [7]:
# ---- SQL QUERIES FOR POWER BI ----

# 1. Funnel Summary
funnel = pd.read_sql("""
    SELECT 
        event_type,
        COUNT(*) as total_events,
        COUNT(DISTINCT user_id) as unique_users
    FROM ecommerce_events
    GROUP BY event_type
""", engine)
print("=== Funnel Summary ===")
print(funnel)

# 2. Brand Performance
brand_perf = pd.read_sql("""
    SELECT 
        brand,
        event_type,
        COUNT(*) as total
    FROM ecommerce_events
    WHERE brand != 'unknown'
    GROUP BY brand, event_type
    ORDER BY total DESC
    LIMIT 30
""", engine)
print("\n=== Brand Performance ===")
print(brand_perf)

# 3. Hourly Traffic
hourly = pd.read_sql("""
    SELECT 
        hour,
        COUNT(*) as total_events,
        SUM(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) as purchases
    FROM ecommerce_events
    GROUP BY hour
    ORDER BY hour
""", engine)
print("\n=== Hourly Traffic ===")
print(hourly)

# 4. Category Performance
category = pd.read_sql("""
    SELECT 
        cat_level1,
        event_type,
        COUNT(*) as total,
        ROUND(AVG(price), 2) as avg_price
    FROM ecommerce_events
    WHERE cat_level1 != 'unknown'
    GROUP BY cat_level1, event_type
    ORDER BY total DESC
""", engine)
print("\n=== Category Performance ===")
print(category)

# Save all as CSVs for Power BI
funnel.to_csv("../data/funnel.csv", index=False)
brand_perf.to_csv("../data/brand_performance.csv", index=False)
hourly.to_csv("../data/hourly_traffic.csv", index=False)
category.to_csv("../data/category_performance.csv", index=False)

print("\nAll CSVs saved!")


=== Funnel Summary ===
  event_type  total_events  unique_users
0       cart          7763          4501
1   purchase          9595          7389
2       view        482642         92743

=== Brand Performance ===
       brand event_type  total
0    samsung       view  57015
1      apple       view  45838
2     xiaomi       view  34611
3     huawei       view  11111
4    lucente       view   7495
5       oppo       view   6273
6   cordiant       view   6107
7         lg       view   5995
8      bosch       view   5705
9       sony       view   5258
10      acer       view   4993
11     artel       view   4296
12    lenovo       view   3564
13   respect       view   3481
14        hp       view   3330
15   indesit       view   3275
16  dauscher       view   3216
17  triangle       view   3197
18        sv       view   3097
19   redmond       view   2955
20   philips       view   2917
21      asus       view   2825
22       brw       view   2802
23   samsung       cart   2770
24     vite